In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/programmer3/secure-healthcare-iot-monitoring-dataset/Patient_Dataset.csv


In [2]:
# ==================== CELL 1 ====================
# Imports
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.backend import clear_session
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [3]:
# ==================== CELL 2 ====================
# Step 1: Load Dataset
df = pd.read_csv("/kaggle/input/datasets/programmer3/secure-healthcare-iot-monitoring-dataset/Patient_Dataset.csv")

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (2000, 10)
  Patient ID            Timestamp  Heart Rate (bpm)  Temperature (°C)  \
0      P1000  2025-05-06 08:16:48                98              36.5   
1      P1001  2025-05-06 08:06:51                88              37.5   
2      P1002  2025-05-06 08:57:35                74              36.5   
3      P1003  2025-05-06 08:20:47                67              36.8   
4      P1004  2025-05-06 08:35:05                80              39.0   

  Blood Pressure (mmHg) Device ID       IP Address    Access Type  \
0                116/84    Dev-E4   192.168.70.147       Web - PC   
1                114/75    Dev-K7   192.168.46.103   App - Tablet   
2                141/99    Dev-O7  192.168.124.118   App - Mobile   
3                114/67    Dev-W2  192.168.128.103   App - Mobile   
4                114/77    Dev-C2  192.168.169.211  Web - Unknown   

            Action  Target  
0      Data Upload       0  
1      Data Upload       0  
2      Data Upload       0  
3   

In [4]:
# ==================== CELL 3 ====================
# Step 2: Split Blood Pressure
df[['Systolic', 'Diastolic']] = df['Blood Pressure (mmHg)'].str.split('/', expand=True)
df['Systolic'] = pd.to_numeric(df['Systolic'])
df['Diastolic'] = pd.to_numeric(df['Diastolic'])
df.drop('Blood Pressure (mmHg)', axis=1, inplace=True)

print(df.head())

  Patient ID            Timestamp  Heart Rate (bpm)  Temperature (°C)  \
0      P1000  2025-05-06 08:16:48                98              36.5   
1      P1001  2025-05-06 08:06:51                88              37.5   
2      P1002  2025-05-06 08:57:35                74              36.5   
3      P1003  2025-05-06 08:20:47                67              36.8   
4      P1004  2025-05-06 08:35:05                80              39.0   

  Device ID       IP Address    Access Type           Action  Target  \
0    Dev-E4   192.168.70.147       Web - PC      Data Upload       0   
1    Dev-K7   192.168.46.103   App - Tablet      Data Upload       0   
2    Dev-O7  192.168.124.118   App - Mobile      Data Upload       0   
3    Dev-W2  192.168.128.103   App - Mobile      Data Upload       0   
4    Dev-C2  192.168.169.211  Web - Unknown  Alert Triggered       1   

   Systolic  Diastolic  
0       116         84  
1       114         75  
2       141         99  
3       114         67  
4  

In [5]:
# ==================== CELL 4 ====================
# Step 3: Train-Test Split FIRST (Prevent Leakage)
from sklearn.model_selection import train_test_split

X = df.drop(['Target', 'Patient ID', 'Timestamp', 'Device ID', 'IP Address'], axis=1)
y = df['Target']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Train: (1400, 6)
Val: (300, 6)
Test: (300, 6)


In [6]:
 # ==================== CELL 5 ====================
# Step 4: Preprocessing (Only on Train → No Leakage)
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# One-Hot Encoding
cat_cols = ['Access Type', 'Action']

# Perform encoding on training data
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)

# Encode validation and test data, aligning them to X_train columns to prevent missing columns
X_val = pd.get_dummies(X_val, columns=cat_cols, drop_first=True).reindex(columns=X_train.columns, fill_value=0)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True).reindex(columns=X_train.columns, fill_value=0)

# MinMax Scaling
num_features = ['Heart Rate (bpm)', 'Temperature (°C)', 'Systolic', 'Diastolic']

scaler = MinMaxScaler()
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_val[num_features] = scaler.transform(X_val[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

print("✅ Preprocessing completed without leakage")
print(X_train.head())

✅ Preprocessing completed without leakage
      Heart Rate (bpm)  Temperature (°C)  Systolic  Diastolic  \
567              0.675          0.200000  0.600000      0.475   
779              0.550          0.885714  1.000000      0.350   
321              0.575          0.285714  0.033333      1.000   
1509             0.400          0.457143  0.833333      0.600   
732              0.375          0.428571  0.183333      0.675   

      Access Type_App - Tablet  Access Type_Web - PC  \
567                      False                  True   
779                      False                 False   
321                      False                 False   
1509                     False                 False   
732                      False                  True   

      Access Type_Web - Unknown  Action_Data Upload  Action_Unauthorized Login  
567                       False                True                      False  
779                       False                True                 

In [7]:
# ==================== CELL 7 ====================
# Step 6: Create Sequences
import numpy as np

SEQ_LEN = 20

def create_sequences(data, labels, seq_len=20):
    X_seq = []
    y_seq = []
    data = data.astype(np.float32)
    for i in range(len(data) - seq_len):
        X_seq.append(data[i:i + seq_len])
        y_seq.append(1 if 1 in labels[i:i + seq_len] else 0)
    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(X_train.values, y_train.values)
X_val_seq, y_val_seq = create_sequences(X_val.values, y_val.values)
X_test_seq, y_test_seq = create_sequences(X_test.values, y_test.values)

print("Train sequences:", X_train_seq.shape)
print("Val sequences:", X_val_seq.shape)
print("Test sequences:", X_test_seq.shape)

Train sequences: (1380, 20, 9)
Val sequences: (280, 20, 9)
Test sequences: (280, 20, 9)


In [8]:
# ==================== CELL 8 ====================
# Step 7: Power Analysis
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()
required_n = analysis.solve_power(effect_size=0.5, power=0.80, alpha=0.05, ratio=1.0)

print("=== Power Analysis ===")
print(f"Minimum required sequences per group: {int(required_n)}")
print(f"Our training sequences: {len(X_train_seq)} → Sufficient")

=== Power Analysis ===
Minimum required sequences per group: 63
Our training sequences: 1380 → Sufficient


In [9]:
# ==================== CELL 9 ====================
# Step 8: Model Definition
def build_lstm_autoencoder(timesteps, features):
    inputs = Input(shape=(timesteps, features))
    encoded = LSTM(128, activation='relu', return_sequences=True)(inputs)
    encoded = LSTM(64, activation='relu', return_sequences=False)(encoded)
    latent = RepeatVector(timesteps)(encoded)
    decoded = LSTM(64, activation='relu', return_sequences=True)(latent)
    decoded = LSTM(128, activation='relu', return_sequences=True)(decoded)
    outputs = TimeDistributed(Dense(features))(decoded)
    
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
 import gc
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.backend import clear_session
import numpy as np
import pandas as pd

# ⚡ KONTROL SWITCH: Har dafa Kaggle RESTART karne ke baad yahan values badlein
CURRENT_SEED = 777
CURRENT_EPSILON = 5.0  # Isko bari bari badalna hai (0.1, 0.5, 1.0, 2.0, 5.0)

# --- Full Research Parameters ---
NUM_CLIENTS = 5
FL_ROUNDS = 5
LOCAL_EPOCHS = 8
BATCH_SIZE = 64
N_SPLITS = 5
CLIP_NORM = 1.0
# --------------------------------

early_stop = EarlyStopping(monitor='loss', patience=2, restore_best_weights=True)
timesteps = X_train_seq.shape[1]
features = X_train_seq.shape[2]

print(f"🚀 STARTING MICRO-RUN: SEED = {CURRENT_SEED} | ε = {CURRENT_EPSILON}")
print("="*60)

# CSV File setup for Auto-Save
file_name = f'dp_results_seed_{CURRENT_SEED}_eps_{CURRENT_EPSILON}.csv'
if os.path.exists(file_name):
    os.remove(file_name) # Pura run naye sirey se shuru ho to purani file hata de

noise_multiplier = 0.01 / CURRENT_EPSILON
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=CURRENT_SEED)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y_train_seq)), y_train_seq)):
    X_fold_train = X_train_seq[train_idx]
    X_fold_val = X_train_seq[val_idx]
    y_fold_val = y_train_seq[val_idx]
    
    chunk_size = len(X_fold_train) // NUM_CLIENTS
    client_data = [X_fold_train[i * chunk_size : len(X_fold_train) if i == NUM_CLIENTS - 1 else (i + 1) * chunk_size] for i in range(NUM_CLIENTS)]
    
    clear_session()
    global_model = build_lstm_autoencoder(timesteps, features)
    
    for round_num in range(FL_ROUNDS):
        global_weights = global_model.get_weights()
        client_weights = []
        
        for i in range(NUM_CLIENTS):
            local_model = build_lstm_autoencoder(timesteps, features)
            local_model.set_weights(global_weights)
            local_model.fit(client_data[i], client_data[i], epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=[early_stop])
            
            local_w = local_model.get_weights()
            noisy_w = []
            for w in local_w:
                clipped_w = np.clip(w, -CLIP_NORM, CLIP_NORM)
                noise = np.random.normal(loc=0.0, scale=noise_multiplier, size=w.shape)
                noisy_w.append(clipped_w + noise)
            client_weights.append(noisy_w)
            
        new_global_weights = [np.mean(np.array(weights), axis=0) for weights in zip(*client_weights)]
        global_model.set_weights(new_global_weights)
        
    # Evaluation
    X_val_pred = global_model.predict(X_fold_val, verbose=0)
    mse = np.mean(np.power(X_fold_val - X_val_pred, 2), axis=(1,2))
    mse = np.nan_to_num(mse, nan=0.0, posinf=1e6, neginf=0.0)
    
    prec, rec, thresh = precision_recall_curve(y_fold_val, mse)
    f1_scores = 2 * prec * rec / (prec + rec + 1e-10)
    best_thresh = thresh[np.argmax(f1_scores[:-1])] if len(thresh) > 0 else np.mean(mse)
    
    y_pred = (mse > best_thresh).astype(int)
    
    # ⚡ AUTO-SAVE AFTER EVERY FOLD
    fold_result = pd.DataFrame([{
        'Seed': CURRENT_SEED, 'Epsilon': CURRENT_EPSILON, 'Fold': fold + 1,
        'Accuracy': accuracy_score(y_fold_val, y_pred),
        'Precision': precision_score(y_fold_val, y_pred, zero_division=0),
        'Recall': recall_score(y_fold_val, y_pred, zero_division=0),
        'F1_Score': f1_score(y_fold_val, y_pred, zero_division=0),
        'AUC': roc_auc_score(y_fold_val, mse)
    }])
    
    # Append to CSV (header sirf pehli dafa likha jayega)
    fold_result.to_csv(file_name, mode='a', index=False, header=not os.path.exists(file_name))
    
    # Strict Memory Cleanup
    clear_session()
    del global_model
    del client_weights
    del client_data
    del X_fold_train
    del X_fold_val
    gc.collect()
    
    print(f"  ✅ Fold {fold + 1}/5 Done & Saved! | RAM Cleared!")

print(f"\n🎉 Micro-Run Complete! Data saved to '{file_name}'.")
print("🛑 PLEASE RESTART KAGGLE SESSION BEFORE CHANGING EPSILON.")

🚀 STARTING MICRO-RUN: SEED = 777 | ε = 5.0


I0000 00:00:1783404042.799690      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783404042.802818      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1783404054.172995     130 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  ✅ Fold 1/5 Done & Saved! | RAM Cleared!
  ✅ Fold 2/5 Done & Saved! | RAM Cleared!
